# Decision Tree with Scikit-Learn - Day 5

## Exact Splitter for Binary Classification with Numerical Features

In this unit, we explore the simplest and most common splitter algorithm, which creates conditions of the form (feature >= threshold) for binary classification tasks.

**Setting:**
- Binary classification task
- No missing values in the examples
- Numerical features only

---

## Problem Definition

Assume a set of N examples with a numerical feature and a binary label "orange" and "blue". Formally, let's describe the dataset D as:

D = {(x_i, y_i)} for i = 1 to N

where:
- x_i is the value of a numerical feature in R (real numbers)
- y_i is a binary classification label value in {orange, blue}

**Objective:** Find a threshold value t such that dividing the examples D into groups D_left (x < t) and D_right (x >= t) improves the separation of the labels.

---

## Shannon Entropy

Shannon entropy is a measure of disorder. For a binary label:

| Condition | Entropy |
|-----------|---------|
| Balanced labels (50% blue, 50% orange) | Maximum |
| Pure labels (100% blue or 100% orange) | Minimum (zero) |

---

## Information Gain

We want to find a condition that decreases the weighted sum of the entropy of the label distributions in D_left and D_right. The corresponding score is the information gain, which is the difference between D's entropy and {D_left, D_right} entropy.

**Formula:**

IG(T, D) = H(D) - (|D_left|/|D| * H(D_left) + |D_right|/|D| * H(D_right))

where:
- IG(T, D) is the information gain
- H(D) is the entropy of set D
- |D| is the number of elements in set D
- T is the threshold value

**Properties:**
- Information gain is always positive or null
- Converges to zero as threshold approaches min/max values
- Zero when D_left ratio equals D_right ratio equals D ratio

---

## Key Observations

**Candidate Threshold Values:**
- The candidate values for t in real numbers are infinite
- However, given finite examples, only a finite number of divisions exist
- Only a finite number of t values are meaningful to test

**Classical Approach:**
1. Sort values x_i in increasing order
2. Test t for every value halfway between consecutive sorted values

Example: If sorted values are 8.5 and 8.7, test t = 8.6

**Time Complexity:** O(N log N) per node (due to sorting)

**Overall Training Complexity:** O(m * N log N)
where m is number of features and N is number of training examples

**Important:** Values of features do not matter, only the order matters. Therefore, we do NOT need to normalize or scale numerical features when training a decision tree.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

In [ ]:
# Dataset: Numerical feature with binary labels
X = np.array([18, 22, 25, 30, 35, 40, 45, 50, 55, 60]).reshape(-1, 1)
# Labels (1 = blue, 0 = orange)
y = np.array([0, 0, 0, 1, 1, 1, 0, 0, 0, 0])

print("Dataset:")
print("="*40)
for i in range(len(X)):
    color = "blue" if y[i] == 1 else "orange"
    print(f"x={X[i][0]}, y={color}")

In [ ]:
# Train Decision Tree with entropy criterion (information gain)
clf = DecisionTreeClassifier(criterion='entropy', max_depth=1)
clf.fit(X, y)

print("\nBest split threshold:", clf.tree_.threshold[0])
print("Feature used:", clf.tree_.feature[0])
print("Information gain:", clf.tree_.impurity[0] - clf.tree_.impurity[1] - clf.tree_.impurity[2])

In [ ]:
# Visualize the split
from sklearn.tree import plot_tree

plt.figure(figsize=(10, 6))
plot_tree(clf, feature_names=['Feature'], 
          class_names=['orange', 'blue'], 
          filled=True, rounded=True)
plt.title("Decision Tree with Best Split")
plt.show()

In [ ]:
# Analyze the split
threshold = clf.tree_.threshold[0]

left_mask = X.flatten() < threshold
right_mask = X.flatten() >= threshold

print("\nSplit Analysis:")
print("="*50)
print(f"Threshold: {threshold:.2f}")
print(f"Left side (x < {threshold:.2f}): {X[left_mask].flatten()}")
print(f"  Blues: {np.sum(y[left_mask])}, Oranges: {len(y[left_mask]) - np.sum(y[left_mask])}")
print(f"Right side (x >= {threshold:.2f}): {X[right_mask].flatten()}")
print(f"  Blues: {np.sum(y[right_mask])}, Oranges: {len(y[right_mask]) - np.sum(y[right_mask])}")
print("="*50)

In [ ]:
# Calculate entropy and information gain manually
def entropy_calc(y):
    if len(y) == 0:
        return 0
    p = np.mean(y)
    if p == 0 or p == 1:
        return 0
    return -p * np.log2(p) - (1-p) * np.log2(1-p)

parent_entropy = entropy_calc(y)
left_entropy = entropy_calc(y[left_mask])
right_entropy = entropy_calc(y[right_mask])

w_left = len(y[left_mask]) / len(y)
w_right = len(y[right_mask]) / len(y)

info_gain = parent_entropy - (w_left * left_entropy + w_right * right_entropy)

print(f"\nParent Entropy: {parent_entropy:.4f}")
print(f"Left Entropy: {left_entropy:.4f}")
print(f"Right Entropy: {right_entropy:.4f}")
print(f"Information Gain: {info_gain:.4f}")

In [ ]:
# Try different thresholds
print("\nAll Candidate Thresholds:")
print("="*60)
print(f"{'Threshold':<12} {'Left Blues':<12} {'Right Blues':<13} {'Info Gain':<12}")
print("-"*60)

X_sorted = np.sort(X.flatten())

for i in range(len(X_sorted)-1):
    t_test = (X_sorted[i] + X_sorted[i+1]) / 2
    left = y[X.flatten() < t_test]
    right = y[X.flatten() >= t_test]
    
    ig_test = entropy_calc(y) - (len(left)/len(y)*entropy_calc(left) + len(right)/len(y)*entropy_calc(right))
    
    print(f"{t_test:<12.1f} {np.sum(left):<12} {np.sum(right):<13} {ig_test:<12.4f}")

print("="*60)

In [ ]:
# Visualize the data and split
plt.figure(figsize=(10, 4))

# Plot data points
colors = ['orange' if label == 0 else 'blue' for label in y]
plt.scatter(X, np.zeros_like(X), c=colors, s=100, edgecolors='black', zorder=3)

# Plot threshold line
plt.axvline(x=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.2f}')

# Add labels
for i, (x_val, y_val) in enumerate(zip(X.flatten(), y)):
    label = 'blue' if y_val == 1 else 'orange'
    plt.annotate(label, (x_val, 0.02), ha='center', fontsize=9)

plt.xlabel('Feature Value')
plt.title('Best Split for Binary Classification')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Day 5 Completed
print("\n" + "="*50)
print("DAY 5 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Exact splitter for binary classification")
print("- Shannon entropy for binary labels")
print("- Information gain calculation")
print("- Finding best threshold")
print("- Why no feature scaling needed for decision trees")
print("="*50)"
print("\nNext: Day 6 - Overfitting and Pruning")